# Game Simulation

Second part of the project where each model receives the original rulebook and a game situation, and suggests the best next move.

In [ ]:
!pip install -q bitsandbytes accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
basePath = "/content/drive/My Drive/Progetto NLP 2026"
for folder in ["src", "data"]:
    path = os.path.join(basePath, folder)
    if path not in sys.path:
        sys.path.insert(0, path)
outPath = os.path.join(basePath, "results", "Results_simulation")
os.makedirs(outPath, exist_ok=True)

Mounted at /content/drive


In [ ]:
import json, gc
import torch
from tqdm.auto import tqdm

from models import Qwen, Llama1B, Mistral7B, Qwen7B
from dataset import games
from dataset_simulation import simulations
from runTest import testGameSimulation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

using device: cuda


## Configuration

In [ ]:
models = [
    ("Qwen", Qwen),
    ("Llama-1B", Llama1B),
    ("Mistral-7B", Mistral7B),
    ("Qwen-7B", Qwen7B),
]
runs = 3
baseSeed = 42
maxNewTokens = 1500
selectedGames = [
    "connect-four", "checkers", "nine-mens-morris",
    "ticket-to-ride", "kingdomino", "azul",
    "uno", "exploding-kittens", "mao",
    "monopoly", "clue", "forbidden-island",
    "werewolf", "dixit", "the_resistance",
]

## Generation

In [ ]:
def resultsPath(modelName):
    return os.path.join(outPath, f"results_{modelName.replace('-', '_')}.json")

# results are rewritten after every answer, through a temporary file,
# so that an interrupted session loses nothing
def save(path, rows):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)

for modelName, modelClass in models:
    path = resultsPath(modelName)

    # answers already saved are skipped
    rows = json.load(open(path, encoding="utf-8")) if os.path.exists(path) else []
    done = {(r["game"], r["iteration"]) for r in rows}
    todo = [(game, run) for game in selectedGames for run in range(1, runs + 1) if (game, run) not in done]
    if not todo:
        print(modelName, ": done"); continue

    model = modelClass(device)
    print(f"{modelName}: max context {model.max_context} token, {len(todo)} to do")

    for gameName, run in tqdm(todo, desc=modelName):
        seed = baseSeed + run # 43, 44, 45: one seed per iteration
        output = testGameSimulation(model, gameName, games[gameName], simulations[gameName]["stage"],
                                    max_new_tokens=maxNewTokens, seed=seed)
        rows.append({"model": modelName, "iteration": run, **output})
        save(path, rows)
        torch.cuda.empty_cache()

    # free the GPU memory before loading the next model
    del model; gc.collect(); torch.cuda.empty_cache()

Qwen : done
Llama-1B : done
Mistral-7B : done
Qwen-7B : done


## Quick check
How many answers stopped at the `max_new_tokens` limit (answer cut off).

In [ ]:
import pandas as pd

rows = []
for fileName in sorted(os.listdir(outPath)):
    if fileName.endswith(".json"):
        rows.extend(json.load(open(os.path.join(outPath, fileName), encoding="utf-8")))
df = pd.DataFrame(rows)

print(df.groupby("model").size())
print("\nmax_new_tokens limit exceeded:")
print(df.groupby("model")["hit_max_new_tokens"].mean().round(2))

model
Llama-1B      45
Mistral-7B    45
Qwen          45
Qwen-7B       45
dtype: int64

max_new_tokens limit exceeded:
model
Llama-1B      0.0
Mistral-7B    0.0
Qwen          0.0
Qwen-7B       0.0
Name: hit_max_new_tokens, dtype: float64
